In [6]:
# 爬京东上华为手机的数据
import requests
import json
import time

def get_jd_reviews(product_id, max_pages=3):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36',
        'Referer': f'https://item.jd.com/{product_id}.html',
        'Cookie': "bceef990357d47e99a2220c264653447,3,984773",
        'Host': 'club.jd.com'
    }

    reviews = []

    for page in range(max_pages):
        print(f"正在抓取第 {page + 1} 页...")
        # score=0表示全部评价，sortType=5表示推荐排序
        url = f'https://club.jd.com/comment/productPageComments.action?productId={product_id}&score=0&sortType=5&page={page}&pageSize=10&isShadowSku=0&fold=1'

        try:
            response = requests.get(url, headers=headers)
            response.raise_for_status()

            # 尝试解析 JSON 数据
            data = response.json()
            comments = data.get('comments', [])

            if not comments:
                print("没有更多评价了。")
                break

            for comment in comments:
                content = comment.get('content', '')
                creation_time = comment.get('creationTime', '')
                score = comment.get('score', '')
                reviews.append({
                    '时间': creation_time,
                    '评分': score,
                    '内容': content
                })
                print(f"[{creation_time}] 评分:{score}星 - {content[:30]}...")

            # 延时，防止请求过快被京东封禁 IP
            time.sleep(2)

        except requests.exceptions.RequestException as e:
            print(f"请求发生错误: {e}")
            break
        except json.JSONDecodeError:
            print("解析 JSON 失败，可能触发了反爬机制（需要登录或验证码）。")
            break

    return reviews

if __name__ == '__main__':
    # 你提供的商品 ID
    ITEM_ID = '10199121936185'
    # 抓取前 3 页评价作为示例
    get_jd_reviews(ITEM_ID, max_pages=3)


正在抓取第 1 页...
请求发生错误: Expecting value: line 1 column 1 (char 0)
